# Predictive Modeling Using Machine Learning
## Loan Approval Prediction (Classification) + Interest-Rate Regression

End-to-end supervised ML pipeline:
**Raw Data -> Cleaning -> Feature Engineering -> Encode/Scale -> Train/Test Split -> Model Training -> Hyperparameter Tuning -> Evaluation -> Visualization -> Prediction**

Run the cells top-to-bottom. The project also ships reusable source modules in `src/`.

## 0. Setup & Imports

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
sys.path.append(str(Path('..') / 'src'))

import config as cfg
from preprocessing import (load_data, prepare_pipeline, encode_categoricals)
from feature_engineering import create_features, add_polynomial_features, select_k_best
from train_model import (train_classification, train_regression,
                         select_best_classifier, save_model)
from evaluate import (evaluate_classifier, evaluate_regression,
                      classification_metrics, regression_metrics)
from pipeline import run_full_pipeline

print('Environment ready. Project root:', cfg.BASE)

Environment ready. Project root: D:\roopank\coding\t\Predictive Modeling Using Machine Learning


## 1. Load Raw Data

In [2]:
train_raw, test_raw = load_data()
print('train:', train_raw.shape, '| test:', test_raw.shape)
train_raw.head()

train: (4160, 15) | test: (1000, 15)


,age,income,employment_years,loan_amount,loan_term,credit_score,existing_loans,dependents,gender,education,home_ownership,loan_purpose,marital_status,interest_rate,loan_approved
0,51.2,128862.2,22.7,23021.7,77.2,425.8,3.4,0.9,Female,Master,Mortgage,Education,Single,7.055,0
1,35.0,151332.6,29.9,25981.9,294.1,655.2,5.5,1.8,Female,Bachelor,Own,Business,Married,2.989,1
2,56.5,129566.2,29.7,15574.0,155.1,569.8,3.4,1.6,Female,High School,Own,Home,Married,4.163,1
3,58.8,198359.1,16.4,NaN,177.3,521.0,4.5,2.5,Male,PhD,Own,Education,Married,5.465,0
4,25.0,170042.6,21.6,26958.4,205.7,552.4,3.8,2.6,Male,Master,Own,Personal,Single,4.708,1


### Quick EDA

In [3]:
print('Missing values per column:')
print(train_raw.isna().sum()[train_raw.isna().sum() > 0])
print('\nDuplicate rows:', train_raw.duplicated().sum())
print('\nTarget balance:')
print(train_raw['loan_approved'].value_counts(normalize=True).round(3))

num_cols = cfg.BASE_NUMERIC_COLS
train_raw[num_cols].describe().round(2)

Missing values per column:
income              185
employment_years    207
loan_amount         215
credit_score        209
education           213
dtype: int64

Duplicate rows: 160

Target balance:
loan_approved
1    0.532
0    0.468
Name: proportion, dtype: float64


,age,income,employment_years,loan_amount,loan_term,credit_score,existing_loans,dependents
count,4160.00,3975.00,3953.00,3945.00,4160.00,3951.00,4160.00,4160.00
mean,47.27,132691.45,19.89,28874.08,187.90,576.25,4.04,2.50
std,11.39,88441.32,7.94,23076.59,78.89,91.66,1.51,1.16
min,25.00,20000.00,0.00,1000.00,6.00,300.00,0.00,0.00
25%,39.10,96949.30,14.60,20234.50,133.28,515.65,3.00,1.70
50%,47.50,120181.50,19.80,25533.30,188.10,575.80,4.10,2.50
75%,55.20,145814.05,25.20,31589.70,242.20,636.80,5.10,3.30
max,70.00,1052153.68,40.00,331604.83,360.00,850.00,8.00,5.00


## 2. Preprocessing
Handles duplicates, imputes missing values (using **training** statistics), removes IQR outliers from the training set only, one-hot encodes categoricals, and scales with `StandardScaler` (fitted on train only).

In [4]:
(X_train, X_val, X_test,
 y_train, y_val, y_test,
 feature_cols, scaler) = prepare_pipeline(
    train_raw, test_raw, test_size=0.2, random_state=cfg.RANDOM_STATE)

print('Scaled train/val/test shapes:', X_train.shape, X_val.shape, X_test.shape)
print('Number of features:', len(feature_cols))
from pathlib import Path
(cfg.MODELS_DIR / 'features.json').write_text(__import__('json').dumps(feature_cols))

[clean] removed 160 duplicate rows (kept 4000)
[impute] saved stats -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\models\imputation_stats.json
[impute] numeric 'income': filled 181 missing with 120257.8
[impute] numeric 'employment_years': filled 199 missing with 19.8
[impute] numeric 'loan_amount': filled 203 missing with 25576.1
[impute] numeric 'credit_score': filled 201 missing with 576.4
[impute] categorical 'education': filled 206 missing with Bachelor


[outliers] removed 269 rows via IQR (kept 3731)
[impute] numeric 'income': filled 46 missing with 120257.8
[impute] numeric 'employment_years': filled 45 missing with 19.8
[impute] numeric 'loan_amount': filled 59 missing with 25576.1
[impute] numeric 'credit_score': filled 52 missing with 576.4
[impute] categorical 'education': filled 57 missing with Bachelor


[encode] one-hot encoded 5 categorical columns -> 34 total columns
[encode] one-hot encoded 5 categorical columns -> 34 total columns


[scale] fitted StandardScaler on training data
[scale] saved scaler -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\models\scaler.pkl
Scaled train/val/test shapes: (2984, 33) (747, 33) (1000, 33)
Number of features: 33


639

## 3. Feature Engineering
New meaningful features are created before encoding (leakage-safe, row-wise):
`debt_to_income`, `loan_to_income`, `income_per_dependent`, `loan_per_month`, `employment_ratio`, `credit_tier`, `age_group`.

In [5]:
sample = create_features(train_raw.head(5))
print(sample[['age','income','dependents','income_per_dependent','credit_score','credit_tier','age_group']])

    age    income  dependents  income_per_dependent  credit_score  \
0  51.2  128862.2         0.9          67822.210526         425.8   
1  35.0  151332.6         1.8          54047.357143         655.2   
2  56.5  129566.2         1.6          49833.153846         569.8   
3  58.8  198359.1         2.5          56674.028571         521.0   
4  25.0  170042.6         2.6          47234.055556         552.4   

   credit_tier  age_group  
0            0          3  
1            1          1  
2            0          3  
3            0          3  
4            0          0  


### (Optional) Polynomial Features
Demonstrates interaction terms via `PolynomialFeatures` (fit on train only).

In [6]:
Xtr_p, Xva_p, Xte_p, poly_names, poly = add_polynomial_features(
    X_train, X_val, X_test, feature_cols, degree=2)
print('Original features:', X_train.shape[1], '-> Polynomial features:', Xtr_p.shape[1])

[poly] degree=2 -> 594 features
Original features: 33 -> Polynomial features: 594


### Feature Selection (SelectKBest)

In [7]:
Xtr_k, Xva_k, Xte_k, selector, mask = select_k_best(
    X_train, y_train, X_val, X_test, k=20)
kept = [feature_cols[i] for i in range(len(feature_cols)) if mask[i]]
print('Selected (top 20):', kept)

[select] kept 20 of 33 features
Selected (top 20): ['age', 'income', 'employment_years', 'loan_amount', 'loan_term', 'credit_score', 'existing_loans', 'dependents', 'interest_rate', 'debt_to_income', 'loan_to_income', 'income_per_dependent', 'loan_per_month', 'employment_ratio', 'credit_tier', 'age_group', 'education_High School', 'education_Master', 'home_ownership_Own', 'loan_purpose_Education']


## 4. Model Training & Hyperparameter Tuning
Multiple supervised algorithms are trained with `GridSearchCV` (5-fold cross-validation, scoring=ROC-AUC).

In [8]:
clf_results = train_classification(X_train, y_train, cv=5)
# Per-model best CV scores were printed during training above.

[train] LogisticRegression: best CV roc_auc=0.7776 | params={'C': 0.01}


[train] DecisionTree: best CV roc_auc=0.7300 | params={'max_depth': 5, 'min_samples_split': 2}


[train] RandomForest: best CV roc_auc=0.7705 | params={'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}


[train] KNN: best CV roc_auc=0.7189 | params={'n_neighbors': 15, 'weights': 'distance'}


[train] SVM: best CV roc_auc=0.7648 | params={'estimator__C': 1.0, 'estimator__gamma': 'scale'}
[train] NaiveBayes: best CV roc_auc=0.7580 | params={'var_smoothing': 1e-09}


## 5. Model Selection
Best classifier chosen by validation ROC-AUC, then saved to `models/trained_model.pkl`.

In [9]:
best_name, best_model = select_best_classifier(clf_results, X_val, y_val)
save_model(best_model, 'trained_model.pkl')
print('Saved best model:', best_name)

[select] LogisticRegression: val roc_auc=0.7478
[select] DecisionTree: val roc_auc=0.7123
[select] RandomForest: val roc_auc=0.7332


[select] KNN: val roc_auc=0.7180


[select] SVM: val roc_auc=0.7360
[select] NaiveBayes: val roc_auc=0.7341
[select] BEST classifier = LogisticRegression (val roc_auc=0.7478)
[save] model -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\models\trained_model.pkl
Saved best model: LogisticRegression


## 6. Evaluation on Held-Out Test Set
Generates the confusion matrix, ROC curve, precision-recall curve, feature importance, learning curve (on the training set), and a text classification report in `reports/`.

In [10]:
clf_metrics = evaluate_classifier(best_model, X_test, y_test, feature_cols,
                                  X_train=X_train, y_train=y_train)
clf_metrics

[plot] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\confusion_matrix.png


[plot] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\roc_curve.png


[plot] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\precision_recall_curve.png


[plot] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\feature_importance.png


[plot] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\learning_curve.png
[report] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\classification_report.txt
[metric] accuracy: 0.6840
[metric] precision: 0.7307
[metric] recall: 0.6916
[metric] f1: 0.7106
[metric] roc_auc: 0.7368


{'accuracy': 0.684,
 'precision': 0.7306967984934086,
 'recall': 0.6916221033868093,
 'f1': 0.7106227106227107,
 'roc_auc': 0.7367741463949424}

### Classification Report (text)

In [11]:
print((cfg.REPORTS_DIR / 'classification_report.txt').read_text())

LOAN APPROVAL - CLASSIFICATION REPORT

Key Metrics
------------------------------------------------------------
  accuracy: 0.6840
 precision: 0.7307
    recall: 0.6916
        f1: 0.7106
   roc_auc: 0.7368

Confusion Matrix (rows=true, cols=pred)
------------------------------------------------------------
           Predicted=0  Predicted=1
Actual=0          296         143
Actual=1          173         388

Detailed Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Rejected       0.63      0.67      0.65       439
    Approved       0.73      0.69      0.71       561

    accuracy                           0.68      1000
   macro avg       0.68      0.68      0.68      1000
weighted avg       0.69      0.68      0.68      1000



## 7. Secondary Regression Demo (predict `interest_rate`)
Compares Linear / Decision Tree / Random Forest regressors (metrics: MAE, MSE, RMSE, R²) and saves prediction-vs-actual and residual plots.

In [12]:
(Xr_tr, Xr_va, Xr_te, yr_tr, yr_va, yr_te, reg_feats, _) = prepare_pipeline(
    train_raw, test_raw, target='interest_rate',
    exclude_cols=[], save_scaler=False,
    test_size=0.2, random_state=cfg.RANDOM_STATE, stratify=None)

reg_results = train_regression(Xr_tr, yr_tr, cv=5)
best_reg = max(reg_results.values(),
               key=lambda m: regression_metrics(yr_te, (
                   m.best_estimator_ if hasattr(m,'best_estimator_') else m).predict(Xr_te))['R2'])
best_reg_est = best_reg.best_estimator_ if hasattr(best_reg,'best_estimator_') else best_reg
evaluate_regression(best_reg_est, Xr_te, yr_te)

[clean] removed 160 duplicate rows (kept 4000)
[impute] saved stats -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\models\imputation_stats.json
[impute] numeric 'income': filled 181 missing with 120257.8
[impute] numeric 'employment_years': filled 199 missing with 19.8
[impute] numeric 'loan_amount': filled 203 missing with 25576.1
[impute] numeric 'credit_score': filled 201 missing with 576.4
[impute] categorical 'education': filled 206 missing with Bachelor


[outliers] removed 269 rows via IQR (kept 3731)
[impute] numeric 'income': filled 46 missing with 120257.8
[impute] numeric 'employment_years': filled 45 missing with 19.8
[impute] numeric 'loan_amount': filled 59 missing with 25576.1
[impute] numeric 'credit_score': filled 52 missing with 576.4
[impute] categorical 'education': filled 57 missing with Bachelor


[encode] one-hot encoded 5 categorical columns -> 34 total columns
[encode] one-hot encoded 5 categorical columns -> 34 total columns


[scale] fitted StandardScaler on training data
[train] LinearRegression: mean CV r2=0.8706


[train] DecisionTreeRegressor: best CV r2=0.8208


[train] RandomForestRegressor: best CV r2=0.8574


[plot] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\prediction_vs_actual.png


[plot] saved -> D:\roopank\coding\t\Predictive Modeling Using Machine Learning\reports\residual_plot.png
[metric] MAE: 0.4668
[metric] MSE: 0.5074
[metric] RMSE: 0.7124
[metric] R2: 0.8042


{'MAE': 0.46681532681953736,
 'MSE': 0.5074499590876238,
 'RMSE': np.float64(0.7123552197377541),
 'R2': 0.8041738734728888}

## 8. Prediction on New Records
Loads the saved model + scaler + feature schema and predicts approval for raw inputs.

In [13]:
from predict import predict_new
sample = pd.DataFrame([{
    'age': 35, 'income': 85000, 'employment_years': 10, 'loan_amount': 15000,
    'loan_term': 36, 'credit_score': 720, 'existing_loans': 1, 'dependents': 2,
    'gender': 'Male', 'education': 'Bachelor', 'home_ownership': 'Own',
    'loan_purpose': 'Home', 'marital_status': 'Married'}])
predict_new(sample)

[encode] one-hot encoded 5 categorical columns -> 20 total columns


,approval_probability,prediction
0,0.8911,Approved


## 9. One-Click Full Pipeline
Alternatively, run everything end-to-end from the orchestrator module.

In [14]:
# summary = run_full_pipeline()  # uncomment to reproduce all artifacts